In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


/Users/pratyushsingh/Documents/projects/youtube-rag-application/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
## Document LOADER

In [5]:
yt_api = YouTubeTranscriptApi()
video_id = "SfOaZIGJ_gs"  # Replace
transcript = yt_api.fetch(video_id)

In [6]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Music]', start=0.87, duration=1.82), FetchedTranscriptSnippet(text='[Applause]', start=1.58, duration=18.69), FetchedTranscriptSnippet(text='[Music]', start=2.69, duration=17.58), FetchedTranscriptSnippet(text='uh where Nik', start=21.84, duration=4.16), FetchedTranscriptSnippet(text='>> you told him 5 minutes', start=24.8, duration=2.88), FetchedTranscriptSnippet(text='>> like he has 2 minutes not', start=26.0, duration=4.32), FetchedTranscriptSnippet(text='>> yeah 2 minutes Everything looks good.', start=27.68, duration=5.84), FetchedTranscriptSnippet(text='Just the monitor, the main monitor uh', start=30.32, duration=4.16), FetchedTranscriptSnippet(text='went off.', start=33.52, duration=6.199), FetchedTranscriptSnippet(text=">> Everyone who's done can leave.", start=34.48, duration=5.239), FetchedTranscriptSnippet(text='>> Hi, fam.', start=43.84, duration=2.16), FetchedTranscriptSnippet(text='>> Hey, Nicole.', start=44.879

In [10]:
transcript_text = " ".join(doc.text for doc in transcript)

In [11]:
print(transcript_text)

[Music] [Applause] [Music] uh where Nik >> you told him 5 minutes >> like he has 2 minutes not >> yeah 2 minutes Everything looks good. Just the monitor, the main monitor uh went off. >> Everyone who's done can leave. >> Hi, fam. >> Hey, Nicole. >> How are you? >> I'm good. Sorry I'm late. I got caught up in getting ready for the launch tomorrow and lost track of time and excitement with the final results. But >> no worries. I'm guessing it must be really hectic, right? >> It is a very hectic day. [Music] [Music] I have the model and I've been playing with it a little bit. How is it different, Sam? I'm not an expert at this. So yeah, >> there's all these ways we can talk about, you know, it's better at this metric or it's, you know, can do this amazing coding demo that the, you know, GPT4 couldn't. But the thing that has been most striking for me is in ways that are both big and small, going back from GPT5 to our previous generation model is just so painful. It's just like worse at eve

In [12]:
### Text Splitting

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [13]:
chunks = splitter.split_text(transcript_text)

In [14]:
chunks

["[Music] [Applause] [Music] uh where Nik >> you told him 5 minutes >> like he has 2 minutes not >> yeah 2 minutes Everything looks good. Just the monitor, the main monitor uh went off. >> Everyone who's done can leave. >> Hi, fam. >> Hey, Nicole. >> How are you? >> I'm good. Sorry I'm late. I got caught up in getting ready for the launch tomorrow and lost track of time and excitement with the final results. But >> no worries. I'm guessing it must be really hectic, right? >> It is a very hectic",
 "be really hectic, right? >> It is a very hectic day. [Music] [Music] I have the model and I've been playing with it a little bit. How is it different, Sam? I'm not an expert at this. So yeah, >> there's all these ways we can talk about, you know, it's better at this metric or it's, you know, can do this amazing coding demo that the, you know, GPT4 couldn't. But the thing that has been most striking for me is in ways that are both big and small, going back from GPT5 to our previous generation

In [15]:
len(chunks)

88

In [17]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/pratyushsingh/Documents/projects/youtube-rag-application/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
vectorstore = FAISS.from_texts(chunks, embeddings)

In [19]:
vectorstore

In [21]:
vectorstore.similarity_search("Software engineer?", k=2)

[Document(id='5b1382c3-46ba-436f-8adc-c7b5b6ecc4da', metadata={}, page_content="this is obviously I'm biased because this is like near and dear to my heart, but the fact that as a 25-year-old in India or anywhere else, maybe with a couple of friends, maybe just by yourself, you could use GPT5 to uh help you write the software for a product much more efficiently, help you uh handle customer support, help you write marketing and communications plans, um help you review legal documents, all of these things that would have taken a lot of people and a lot of expertise and you"),
 Document(id='492b2cbd-d833-42fd-8edb-27c2116263c0', metadata={}, page_content='And it is like having PhD level experts in every field available to you 24/7 for whatever you need. not only to ask anything but also to do anything for you. So if you, you know, need a piece of software created, it can kind of do it from scratch all at once. If you need a um if you need a research report on some complicated topic, it ca

In [ ]:
#### Retriever

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [56]:
retriever.invoke("How to start a company?")

[Document(id='1bf5641b-4d4b-44ba-9133-106c5f4b4b23', metadata={}, page_content="which didn't need to be standalone companies or products. But now, as the markets matured, you're really seeing some of these more durable businesses form. So if you lay emphasis on owning the customer almost like the interface with the customer, would you say the relationship gets deeper when I sell a service on top of your model versus a product? Cuz if it's a product company, the the exchange happens once. But if it's a service company, it happens repeatedly and there is room for me to build"),
 Document(id='969e0869-4005-4d84-9031-93eb84123b8e', metadata={}, page_content="that it was very deeply undeserved, but uh grateful that he he said that. Um I mean there's like a lot of people who are starting great companies. We we got lucky here in many ways. We've also worked super hard. uh may maybe the one thing that I would that I think we have done here well is to take a very long time horizon and think ver

In [ ]:
### Augmentation with LLMs

In [37]:
prompt = PromptTemplate(template="""you are a helpful assistant. Use the following retrieved information to answer the question.
If the context is insufficent, say you don't know.
{context}
Question: {question}""",
input_variables=["context", "question"])

In [48]:
question = "How do a sam altman and Nikhil kamath describe their own learning journey with AI in this video?"

In [50]:
context = retriever.invoke(question)

In [51]:
context = " ".join(i.page_content for i in context)


In [52]:
final_prompt = prompt.invoke({'context': context, 'question': question})

In [ ]:
### Generation with LLMs

In [41]:
from langchain_groq import ChatGroq

import dotenv

dotenv.load_dotenv()


True

In [53]:

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # fast + free
    temperature=0.2
)

In [54]:
answer = llm.invoke(final_prompt)

In [55]:
answer

AIMessage(content="I don't have enough information to answer the question. The provided text does not contain a video description, and it appears to be a conversation or interview where the speakers are discussing AI and learning.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 521, 'total_tokens': 561, 'completion_time': 0.110246762, 'completion_tokens_details': None, 'prompt_time': 0.050080626, 'prompt_tokens_details': None, 'queue_time': 0.048428753, 'total_time': 0.160327388}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d2930-67a4-7621-9952-d15124510565-0', usage_metadata={'input_tokens': 521, 'output_tokens': 40, 'total_tokens': 561})